Imports + rutas

In [12]:
from pathlib import Path
import pandas as pd

BASE       = Path("../../")
PEAK_DIR   = BASE / "notebooks" / "00_exploracion" / "peak_analysis" / "outputs"
OUTPUT_DIR = BASE / "dashboard" / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Peak dir:",   PEAK_DIR)
print("Output dir:", OUTPUT_DIR)

Peak dir: ..\..\notebooks\00_exploracion\peak_analysis\outputs
Output dir: ..\..\dashboard\data


Cargar los .parquet

In [13]:
p_H1 = pd.read_parquet(PEAK_DIR / "peak_Hotel1.parquet")
p_H2 = pd.read_parquet(PEAK_DIR / "peak_Hotel2.parquet")
p_H3 = pd.read_parquet(PEAK_DIR / "peak_Hotel3.parquet")
print("Hotel 1:", p_H1.shape)
print("Hotel 2:", p_H2.shape)
print("Hotel 3:", p_H3.shape)

Hotel 1: (10, 13)
Hotel 2: (13, 14)
Hotel 3: (12, 14)


Normalización Hotel 1 (formato evento)

In [14]:
def normalize_event(df, hotel):
    return pd.DataFrame({
        "hotel":                "HOTEL_1" if hotel == "HOTEL_1" else hotel,
        "record_type":          "EVENT",
        "season_type":          "EVENT",
        "direction":            None,
        "start_date":           pd.to_datetime(df["date"], unit="ms"),
        "end_date":             pd.to_datetime(df["date"], unit="ms"),
        "ref_date":             pd.to_datetime(df["date"], unit="ms"),
        "mean_occ":             df["ocup_total"],
        "max_occ":              df["ocup_total"],
        "min_occ":              df["ocup_total"],
        "event_name":           df["evento_real"],
        "event_type":           df["event_date_type"],
        "status_peak":          df["status_peak"],
        "fecha_equivalente_1y": pd.to_datetime(df["fecha_equivalente_1y"], unit="ms", errors="coerce"),
        "real_reference_1y":    df["real_reference_1y"],
        "fecha_equivalente_2y": pd.to_datetime(df["fecha_equivalente_2y"], unit="ms", errors="coerce"),
        "real_reference_2y":    df["real_reference_2y"],
        "segment_id":           None,
    })

Normalización Hotel 2 y Hotel 3 (formato estacional)

In [15]:
def normalize_season(df, hotel):
    return pd.DataFrame({
        "hotel":       hotel,
        "record_type": "SEASON_PHASE",
        "season_type": df["season_type"],
        "direction":   df["regime"],
        "start_date":  pd.to_datetime(df["start"],    unit="ms"),
        "end_date":    pd.to_datetime(df["end"],      unit="ms"),
        "ref_date":    pd.to_datetime(df["mid_date"], unit="ms"),
        "mean_occ":    df["mean_occ"],
        "max_occ":     df["max_occ"],
        "min_occ":     df["min_occ"],
        "event_name":  None,
        "event_type":  None,
        "status_peak": None,
        "segment_id":  df["segment_id"],
    })

Unificación final

In [16]:
H1_n = normalize_event(p_H1,  "HOTEL_1")
H2_n = normalize_season(p_H2, "HOTEL_2")
H3_n = normalize_season(p_H3, "HOTEL_3")

df_peak = pd.concat([H1_n, H2_n, H3_n], ignore_index=True)

Añadir atributos temporales

In [17]:
df_peak["year"]  = df_peak["ref_date"].dt.year
df_peak["month"] = df_peak["ref_date"].dt.month

def month_to_season(m):
    if m in [12, 1, 2]:    return "winter"
    if m in [3, 4, 5]:     return "spring"
    if m in [6, 7, 8, 9]:  return "summer"
    return "autumn"

df_peak["season"] = df_peak["month"].apply(month_to_season)

Export final

In [18]:
out_file = OUTPUT_DIR / "peak_summary.parquet"
df_peak.to_parquet(out_file, index=False)
print("✅ peak_summary.parquet generado correctamente:", out_file)

✅ peak_summary.parquet generado correctamente: ..\..\dashboard\data\peak_summary.parquet
